# Phase 1: Decoder-Only Transformer LM (~101M, RoPE + GQA) for Legal Text

This notebook pretrains a GPT-style language model from scratch on Indian Supreme Court judgments and
generates 128-word continuations for the test prompts, producing a CodaBench `submission.zip`.

**Pipeline:** login -> config -> train a BPE tokenizer -> tokenize the corpus -> build 1024-token blocks
-> train the model with Hugging Face `Trainer` -> evaluate perplexity -> save -> generate test predictions -> zip.

**Before running on Kaggle:**
1. Settings -> Accelerator -> **GPU T4 x2** (or P100)
2. Settings -> **Internet on** (needed to download the dataset from Hugging Face)
3. Add-ons -> Secrets -> add your Hugging Face token under the name **`HF`**
4. Use **Save Version -> Save & Run All (Commit)** so it keeps running after you close the tab.

**Smoke test first:** `SMOKE_TEST = True` in the config cell (the default) runs the whole pipeline at tiny
scale (small tokenizer, 20 training steps, 64 test prompts) in about 10-20 minutes, writing to
`/kaggle/working/cs781_phase1_smoke/`. If it finishes and creates `submission.zip`, set
`SMOKE_TEST = False` and commit again for the real run, whose outputs go to `/kaggle/working/cs781_phase1/`.

## 1. Setup: Imports, Hugging Face Login, GPU Check

All libraries used anywhere in the notebook are imported here once. The Hugging Face token is read from
Kaggle Secrets (name `HF`) and used to download the private course dataset. The GPU check should list at
least one GPU; if it prints `CUDA available: False`, enable the GPU accelerator in the notebook settings.

In [ ]:
import os, json, math, time, random, zipfile, heapq
from collections import Counter, defaultdict
from dataclasses import dataclass, asdict

import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

from huggingface_hub import login, hf_hub_download
from kaggle_secrets import UserSecretsClient
from datasets import Dataset
from transformers import (PreTrainedTokenizer, DataCollatorForLanguageModeling,
                          TrainingArguments, Trainer, TrainerCallback)

HF_TOKEN = UserSecretsClient().get_secret("HF")
os.environ["HF_TOKEN"] = HF_TOKEN
login(token=HF_TOKEN)
print("Hugging Face login successful.")

print("CUDA available:", torch.cuda.is_available())
for i in range(torch.cuda.device_count()):
    print(f"GPU {i}:", torch.cuda.get_device_name(i))

## 2. Configuration

Every tunable setting lives in this one cell:
- **Data:** which train/test files to pull from the Hugging Face dataset repo.
- **BPE tokenizer:** vocabulary size and how many characters of text to learn merges from.
- **Training:** how many tokens to train/validate on, the block (context) length, and a wall-clock
  time limit so the run always finishes inside Kaggle's 12-hour session.
- **Generation:** sampling settings for the 128-word test continuations.
- **Paths:** where the tokenizer, model, plot, predictions and submission zip are saved.

In [ ]:
# Data
HF_DATASET_REPO = "Exploration-Lab/CS-781-Capstone"
TRAIN_FILE = "Phase_1_Pretraining/Train/train_supreme_court.json"   # any of the 4 train files can be used
TEST_FILE = "Phase_1_Pretraining/Test/test.json"

# BPE tokenizer
BPE_VOCAB_SIZE = 50000
BPE_TRAIN_CHARS = 300_000_000

# Training
TRAIN_TOKENS = 200_000_000
VAL_TOKENS = 5_000_000
TRAIN_BLOCK_SIZE = 1024
MODEL_CONTEXT_LENGTH = 1024
EPOCHS = 1
MAX_STEPS = -1                       # -1 = full epoch
EVAL_STEPS = 1000
SAVE_STEPS = 1000
TIME_LIMIT_HOURS = 6.5               # training stops after this, leaving time for generation
SEED = 42
random.seed(SEED)

# Generation
TARGET_GENERATION_WORDS = 128
MAX_GENERATION_TOKENS = 384
TEMPERATURE = 0.1
TOP_P = 0.95
GENERATION_BATCH_SIZE = 32
NUM_TEST_EXAMPLES = None             # None = whole test set

OUTPUT_DIR = "/kaggle/working/cs781_phase1"

# ---- SMOKE TEST: True = tiny end-to-end run (~10-20 min) to check everything works.
# ---- Set to False for the real training run.
SMOKE_TEST = True
if SMOKE_TEST:
    BPE_VOCAB_SIZE, BPE_TRAIN_CHARS = 2000, 2_000_000
    TRAIN_TOKENS, VAL_TOKENS = 300_000, 30_000
    MAX_STEPS, EVAL_STEPS, SAVE_STEPS = 20, 10, 10
    NUM_TEST_EXAMPLES = 64
    OUTPUT_DIR = "/kaggle/working/cs781_phase1_smoke"   # separate folder: smoke files never mix with the real run
    print("*** SMOKE TEST MODE: tiny run, predictions are NOT meant for submission ***")

# Output paths
MODEL_OUTPUT_DIR = os.path.join(OUTPUT_DIR, "legal-dec-rope-101m")
TOKENIZER_DIR = os.path.join(OUTPUT_DIR, "bpe_tokenizer")
BPE_PATH = os.path.join(TOKENIZER_DIR, "bpe_tokenizer.json")
PREDICTION_PATH = os.path.join(OUTPUT_DIR, "prediction.json")
SUBMISSION_ZIP = os.path.join(OUTPUT_DIR, "submission.zip")
LOSS_PLOT_PATH = os.path.join(OUTPUT_DIR, "training_validation_loss.png")
os.makedirs(TOKENIZER_DIR, exist_ok=True)
print("Configuration loaded.")

## 3. Byte-Level BPE Tokenizer (implementation)

`ByteLevelBPE` learns a Byte-Pair-Encoding vocabulary from raw text:
1. Text is split on whitespace into words; each word becomes a sequence of UTF-8 bytes plus an
   end-of-word marker (so any string can be encoded, with no unknown characters).
2. It repeatedly finds the most frequent adjacent pair of symbols and merges it into a new symbol,
   until the vocabulary reaches `BPE_VOCAB_SIZE`. A heap plus incremental pair counts keeps this fast.
3. `tokenize_word` applies the learned merges (in the order they were learned) to encode new text,
   with a per-word cache.

Token ids 0-3 are reserved for `<pad>`, `<unk>`, `<s>` and `</s>`.

In [ ]:
_SHIFT = 20
_MASK = (1 << _SHIFT) - 1
_EOW = _MASK


class ByteLevelBPE:
    def __init__(self, vocab_size=5000):
        self.target_vocab_size = vocab_size
        self.itos = ["<pad>", "<unk>", "<s>", "</s>"]
        self.stoi = {t: i for i, t in enumerate(self.itos)}
        self.merges = []
        self.symbol_to_id = {}
        self._ranks = None
        self._cache = {}

    def fit(self, sentences):
        word_freq = Counter()
        for s in sentences:
            word_freq.update(s.split())

        sym = {}
        key = {}
        for b in range(256):
            i = len(self.itos)
            self.symbol_to_id[(b,)] = i
            self.itos.append([b])
            sym[i] = (b,)
            key[i] = chr(b)
        sym[_EOW] = (256,)
        key[_EOW] = chr(256)

        words, freqs = [], []
        for w, f in word_freq.items():
            words.append([b + 4 for b in w.encode("utf-8")] + [_EOW])
            freqs.append(f)
        del word_freq

        pc = defaultdict(int)
        where = defaultdict(set)
        for wi, w in enumerate(words):
            f = freqs[wi]
            for x, y in zip(w, w[1:]):
                p = (x << _SHIFT) | y
                pc[p] += f
                where[p].add(wi)

        heap = [(-c, key[p >> _SHIFT], key[p & _MASK], p) for p, c in pc.items() if c > 1]
        heapq.heapify(heap)

        itos, merges, s2id = self.itos, self.merges, self.symbol_to_id
        target = self.target_vocab_size
        heappop, heappush = heapq.heappop, heapq.heappush

        while len(itos) < target:
            while heap:
                negc, _, _, best = heappop(heap)
                if pc.get(best, 0) == -negc:
                    break
            else:
                break
            if -negc <= 1:
                break

            a, b = best >> _SHIFT, best & _MASK
            new = len(itos)
            new_sym = sym[a] + sym[b]
            merges.append([list(sym[a]), list(sym[b])])
            s2id[new_sym] = new
            itos.append(list(new_sym))
            sym[new] = new_sym
            key[new] = key[a] + key[b]

            changed = set()
            for wi in where.pop(best, ()):
                w = words[wi]
                f = freqs[wi]
                i = 0
                while i < len(w) - 1:
                    if w[i] == a and w[i + 1] == b:
                        if i > 0:
                            l = w[i - 1]
                            p = (l << _SHIFT) | a; pc[p] -= f; changed.add(p)
                            p = (l << _SHIFT) | new; pc[p] += f; changed.add(p); where[p].add(wi)
                        if i + 2 < len(w):
                            r = w[i + 2]
                            p = (b << _SHIFT) | r; pc[p] -= f; changed.add(p)
                            p = (new << _SHIFT) | r; pc[p] += f; changed.add(p); where[p].add(wi)
                        pc[best] -= f
                        w[i:i + 2] = [new]
                    i += 1

            pc.pop(best, None)
            changed.discard(best)
            for p in changed:
                c = pc[p]
                if c > 1:
                    heappush(heap, (-c, key[p >> _SHIFT], key[p & _MASK], p))
                elif c <= 0:
                    del pc[p]

            if len(merges) % 500 == 0:
                print(f"Merges: {len(merges):,} | Vocab: {len(itos):,} | Best pair count: {-negc:,}")

        self.vocab_size = len(itos)
        self._ranks = None
        self._cache = {}
        print(f"BPE trained: {self.vocab_size:,} vocabulary entries, {len(merges):,} merges")
        return self

    def _build_ranks(self):
        def to_id(t):
            t = tuple(t)
            return _EOW if t == (256,) else self.symbol_to_id[t]
        if not self.symbol_to_id:
            for i, s in enumerate(self.itos):
                if isinstance(s, list):
                    self.symbol_to_id[tuple(s)] = i
        self._ranks = {}
        for l, r in self.merges:
            self._ranks[(to_id(l) << _SHIFT) | to_id(r)] = self.symbol_to_id[tuple(l) + tuple(r)]

    def tokenize_word(self, word):
        out = self._cache.get(word)
        if out is not None:
            return out
        if self._ranks is None:
            self._build_ranks()
        ranks = self._ranks
        w = [b + 4 for b in word.encode("utf-8")]
        w.append(_EOW)

        while len(w) > 1:
            best_new, best_p = None, None
            for x, y in zip(w, w[1:]):
                n = ranks.get((x << _SHIFT) | y)
                if n is not None and (best_new is None or n < best_new):
                    best_new, best_p = n, (x, y)
            if best_new is None:
                break
            a, b = best_p
            i, res = 0, []
            L = len(w)
            while i < L:
                if i < L - 1 and w[i] == a and w[i + 1] == b:
                    res.append(best_new); i += 2
                else:
                    res.append(w[i]); i += 1
            w = res

        out = [1 if t == _EOW else t for t in w]
        self._cache[word] = out
        return out

    def encode_sentence(self, sentence):
        ids = []
        tw = self.tokenize_word
        for word in sentence.split():
            ids.extend(tw(word))
        return ids

    def encode_corpus(self, sentences):
        return [self.encode_sentence(s) for s in sentences]

    def save(self, path):
        with open(path, "w", encoding="utf-8") as f:
            json.dump({"merges": self.merges, "itos": self.itos,
                       "vocab_size": self.vocab_size}, f, ensure_ascii=False)

### Hugging Face tokenizer wrapper

`LegalBPETokenizer` wraps the BPE model as a `PreTrainedTokenizer`, so it works with `Trainer`,
`DataCollatorForLanguageModeling`, `save_pretrained` / `from_pretrained` and `decode`.
`encode_fast` is the fast path used for building the training data and for encoding test prompts.

In [ ]:
_EOW_MARK = "▁"


def _bytes_to_unicode():
    bs = list(range(ord("!"), ord("~") + 1)) + list(range(ord("¡"), ord("¬") + 1)) + list(range(ord("®"), ord("ÿ") + 1))
    cs = bs[:]
    n = 0
    for b in range(256):
        if b not in bs:
            bs.append(b)
            cs.append(256 + n)
            n += 1
    return dict(zip(bs, map(chr, cs)))


class LegalBPETokenizer(PreTrainedTokenizer):
    vocab_files_names = {"vocab_file": "bpe_tokenizer.json"}
    model_input_names = ["input_ids", "attention_mask"]

    def __init__(self, vocab_file, **kwargs):
        with open(vocab_file, "r", encoding="utf-8") as f:
            data = json.load(f)
        bpe = ByteLevelBPE()
        bpe.itos = data["itos"]
        bpe.merges = data["merges"]
        bpe.vocab_size = data["vocab_size"]
        bpe.symbol_to_id = {}
        bpe._build_ranks()
        self.bpe = bpe
        self.eow_id = len(bpe.itos)

        byte_enc = _bytes_to_unicode()
        self.byte_decoder = {c: b for b, c in byte_enc.items()}
        self.id_to_token = list(bpe.itos[:4])
        for sym in bpe.itos[4:]:
            self.id_to_token.append("".join(_EOW_MARK if b == 256 else byte_enc[b] for b in sym))
        self.id_to_token.append(_EOW_MARK)
        self.token_to_id = {t: i for i, t in enumerate(self.id_to_token)}

        kwargs.setdefault("pad_token", "<pad>")
        kwargs.setdefault("unk_token", "<unk>")
        kwargs.setdefault("bos_token", "<s>")
        kwargs.setdefault("eos_token", "</s>")
        kwargs.setdefault("clean_up_tokenization_spaces", False)
        super().__init__(**kwargs)

    def encode_fast(self, text):
        eow, tw = self.eow_id, self.bpe.tokenize_word
        ids = []
        for word in text.split():
            ids.extend(eow if t == 1 else t for t in tw(word))
        return ids

    @property
    def vocab_size(self):
        return len(self.id_to_token)

    def get_vocab(self):
        return dict(self.token_to_id)

    def _tokenize(self, text):
        return [self.id_to_token[i] for i in self.encode_fast(text)]

    def _convert_token_to_id(self, token):
        return self.token_to_id.get(token, self.unk_token_id)

    def _convert_id_to_token(self, index):
        return self.id_to_token[index]

    def convert_tokens_to_string(self, tokens):
        out = bytearray()
        for tok in tokens:
            if tok in ("<pad>", "<unk>", "<s>", "</s>"):
                continue
            for c in tok:
                out.extend(b" " if c == _EOW_MARK else bytes([self.byte_decoder[c]]))
        return out.decode("utf-8", errors="replace").strip()

    def save_vocabulary(self, save_directory, filename_prefix=None):
        path = os.path.join(save_directory, (filename_prefix + "-" if filename_prefix else "") + "bpe_tokenizer.json")
        self.bpe.save(path)
        return (path,)

## 4. Download Training Data

Downloads the training JSON from the Hugging Face dataset repo (cached by `hf_hub_download`).
Each record is a document with a `"text"` field.

In [ ]:
train_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TRAIN_FILE, repo_type="dataset", token=HF_TOKEN)
with open(train_file_path, "r", encoding="utf-8") as f:
    records = json.load(f)
print(f"Loaded {len(records):,} documents from {train_file_path}")

## 5. Train the BPE Tokenizer

Documents are shuffled and fed to the BPE trainer until `BPE_TRAIN_CHARS` characters have been seen.
If a trained tokenizer file already exists, training is skipped. This step runs on the CPU and can
take a while for a 50k vocabulary.

The last lines are a sanity check: they encode a 60-word sample, show how many tokens it needs, and
confirm that decoding gives back exactly the original text (`Round-trip OK: True`).

In [ ]:
def bpe_training_texts(records, max_chars, seed=SEED):
    order = list(range(len(records)))
    random.Random(seed).shuffle(order)
    total = 0
    for idx in order:
        text = records[idx]["text"]
        yield text
        total += len(text)
        if total >= max_chars:
            break
    print(f"BPE training text: {total:,} characters")


if os.path.exists(BPE_PATH):
    print("Found existing BPE tokenizer, skipping training:", BPE_PATH)
else:
    start_time = time.time()
    bpe = ByteLevelBPE(vocab_size=BPE_VOCAB_SIZE)
    bpe.fit(bpe_training_texts(records, BPE_TRAIN_CHARS))
    bpe.save(BPE_PATH)
    print(f"Tokenizer training time: {(time.time() - start_time) / 60:.1f} min")
    print("Saved BPE tokenizer to:", BPE_PATH)

tokenizer = LegalBPETokenizer(vocab_file=BPE_PATH)
print("Vocabulary size:", len(tokenizer))

sample = " ".join(records[0]["text"].split()[:60])
sample_ids = tokenizer.encode_fast(sample)
print(f"{len(sample.split())} words -> {len(sample_ids)} tokens")
print("Round-trip OK:", tokenizer.decode(sample_ids) == sample)

## 6. Tokenize the Corpus into Train / Validation Tokens

Every document is encoded and terminated with an `</s>` (end-of-sequence) token, so the model learns
where documents end. About 5% of documents go to the validation set and the rest to training, until
`VAL_TOKENS` and `TRAIN_TOKENS` are reached.

In [ ]:
train_tokens = []
val_tokens = []

random.seed(SEED)

for data in records:
    text = data["text"]
    tokens = tokenizer.encode_fast(text) + [tokenizer.eos_token_id]

    if random.random() < 0.05:
        if len(val_tokens) < VAL_TOKENS:
            remaining = (VAL_TOKENS - len(val_tokens))
            val_tokens.extend(tokens[:remaining])
    else:
        if len(train_tokens) < TRAIN_TOKENS:
            remaining = (TRAIN_TOKENS - len(train_tokens))
            train_tokens.extend(tokens[:remaining])

    if (len(train_tokens) >= TRAIN_TOKENS and len(val_tokens) >= VAL_TOKENS):
        break

print(f"Train tokens: {len(train_tokens):,}")
print(f"Validation tokens: {len(val_tokens):,}")

## 7. Build Fixed-Length Blocks and Datasets

The token stream is cut into non-overlapping blocks of `TRAIN_BLOCK_SIZE` (1024) tokens. Documents
are packed back to back, so no padding is wasted. The blocks are wrapped in Hugging Face `Dataset`s,
and the raw token lists are then deleted to free RAM.

In [ ]:
def create_blocks(tokens, block_size):
    total_length = (len(tokens) // block_size) * block_size
    tokens = tokens[:total_length]
    return [tokens[i:i + block_size] for i in range(0, total_length, block_size)]

train_dataset = Dataset.from_dict({"input_ids": create_blocks(train_tokens, TRAIN_BLOCK_SIZE)})
val_dataset = Dataset.from_dict({"input_ids": create_blocks(val_tokens, TRAIN_BLOCK_SIZE)})
del train_tokens, val_tokens
print(train_dataset)
print(val_dataset)

## 8. Model Architecture (A-3 Decoder-Only Transformer)

A GPT-style causal language model built from scratch:
- **`LMConfig`**: all architecture hyperparameters.
- **Attention**: `CausalSelfAttention` supports standard multi-head, **grouped-query (GQA)** and
  multi-query attention through `n_kv_heads`, with optional **RoPE** (rotary position embeddings).
  `MultiHeadLatentAttention` (MLA) is also available. The attention itself is our own
  `scratch_attention` (no `nn.MultiheadAttention` / `scaled_dot_product_attention`): `blockwise_attn=True`
  computes softmax(QK^T/sqrt(d))V in blocks of 128 queries with a hand-written backward pass, so the
  T x T score matrix is never stored (same result as the plain path, far less memory).
- **`Block`**: attention + GELU feed-forward, with pre-norm (`norm_first=True`) or post-norm residuals.
- **`DecoderLM`**: token embedding -> N blocks -> final LayerNorm -> output layer tied to the embedding
  matrix (weight tying saves ~38M parameters at a 50k vocab).
- **`A3ForCausalLM`**: a thin wrapper that lets Hugging Face `Trainer` use the model. It shifts the
  labels by one position and returns the next-token cross-entropy loss.
- **`save_a3_model` / `load_a3_model`**: save and load the weights plus config.
- **`_sample_top_p`**: nucleus (top-p) sampling, used during generation.

In [ ]:
# ===== A-3 decoder-only LM, adapted for the capstone =====
# Same architecture as A-3 (Q1 + Q2 + Q3 RoPE). Capstone-only additions are marked "CAPSTONE".


@dataclass
class LMConfig:
    vocab_size: int
    d_model: int = 256
    n_layers: int = 4
    n_heads: int = 4
    n_kv_heads: int = 4
    d_ff: int = 1024
    max_seq_len: int = 256
    dropout: float = 0.1
    pos_emb_type: str = "sinusoidal"   # "sinusoidal" or "rope"
    attention_scale: float = None
    norm_first: bool = False
    attention_type: str = "standard"   # "standard" (MHA/GQA/MQA) or "mla"
    kv_latent_dim: int = None
    blockwise_attn: bool = False       # CAPSTONE: scratch_attention (blockwise), same math, far less memory at T=2048
    rope_base: float = 10000.0         # used when pos_emb_type == "rope"


# ---- From-scratch attention (no nn.MultiheadAttention / F.scaled_dot_product_attention) ----
# out = softmax(q k^T * scale, masked) v, computed in query blocks with the softmax statistics kept per row
# (log-sum-exp), so the full (T, S) score matrix is never stored. The backward pass is written by hand and
# recomputes each block's probabilities from the saved log-sum-exp:
#   P = exp(S - lse),  dV = P^T dO,  dP = dO V^T,  dS = P * (dP - rowsum(dO * O)),  dQ = dS K * scale,  dK = dS^T Q * scale


def _block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset):
    """Scaled, masked scores (B, H, j-i, kend) in fp32 for queries [i, j) against keys [0, kend)."""
    s = torch.matmul(q_blk, k[:, :, :kend].transpose(-2, -1)).float() * scale
    if attn_mask is not None:
        s = s.masked_fill(~attn_mask[..., i:j, :kend], float("-inf"))
    if is_causal:                                           # query row r sits at key position offset + r
        rows = torch.arange(i, j, device=s.device)[:, None] + offset
        s = s.masked_fill(torch.arange(kend, device=s.device)[None, :] > rows, float("-inf"))
    return s


class BlockwiseAttention(torch.autograd.Function):
    """q: (B, H, T, d); k, v: (B, H, S, d); attn_mask: bool, broadcastable to (B, H, T, S), True = may attend.
    Every query row must be allowed at least one key."""

    @staticmethod
    def forward(ctx, q, k, v, attn_mask, is_causal, scale, block):
        T, S = q.size(-2), k.size(-2)
        offset = S - T                                      # causal: the queries are the last T positions
        out = torch.empty_like(q)
        lse = torch.empty(q.shape[:-1], dtype=torch.float32, device=q.device)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S   # causal: later keys are masked anyway, skip them
            s = _block_scores(q[:, :, i:j], k, scale, attn_mask, is_causal, i, j, kend, offset)
            m = s.amax(-1, keepdim=True)
            p = torch.exp(s - m)
            l = p.sum(-1, keepdim=True)
            out[:, :, i:j] = torch.matmul((p / l).to(v.dtype), v[:, :, :kend])
            lse[:, :, i:j] = (m + l.log()).squeeze(-1)
        ctx.save_for_backward(q, k, v, out, lse, attn_mask)
        ctx.is_causal, ctx.scale, ctx.block = is_causal, scale, block
        return out

    @staticmethod
    def backward(ctx, d_out):
        q, k, v, out, lse, attn_mask = ctx.saved_tensors
        is_causal, scale, block = ctx.is_causal, ctx.scale, ctx.block
        T, S = q.size(-2), k.size(-2)
        offset = S - T
        d_out = d_out.contiguous()
        dq = torch.empty_like(q)
        dk = torch.zeros(k.shape, dtype=torch.float32, device=k.device)
        dv = torch.zeros(v.shape, dtype=torch.float32, device=v.device)
        delta = (d_out.float() * out.float()).sum(-1, keepdim=True)        # rowsum(dO * O) = rowsum(P * dP)
        for i in range(0, T, block):
            j = min(i + block, T)
            kend = min(S, offset + j) if is_causal else S
            q_blk, do_blk = q[:, :, i:j], d_out[:, :, i:j]
            p = torch.exp(_block_scores(q_blk, k, scale, attn_mask, is_causal, i, j, kend, offset)
                          - lse[:, :, i:j, None])                           # masked entries -> exactly 0
            dv[:, :, :kend] += torch.matmul(p.to(v.dtype).transpose(-2, -1), do_blk).float()
            dp = torch.matmul(do_blk, v[:, :, :kend].transpose(-2, -1)).float()
            ds = (p * (dp - delta[:, :, i:j])).to(q.dtype)
            dq[:, :, i:j] = torch.matmul(ds, k[:, :, :kend]) * scale
            dk[:, :, :kend] += torch.matmul(ds.transpose(-2, -1), q_blk).float() * scale
        return dq, dk.to(k.dtype), dv.to(v.dtype), None, None, None, None


def scratch_attention(q, k, v, attn_mask=None, is_causal=False, scale=None, block=128):
    """Drop-in for the fused kernel: q (B, H, T, d), k/v (B, H_kv, S, d) with H_kv dividing H (GQA).
    attn_mask: bool (True = may attend), broadcastable to (B, 1, T, S)."""
    scale = q.size(-1) ** -0.5 if scale is None else scale
    B, H, T, d = q.shape
    n_rep = H // k.size(1)
    if n_rep > 1 and not is_causal:
        # GQA without copying K/V: the n_rep query heads of a group become extra query rows for that kv head
        q = q.reshape(B, k.size(1), n_rep * T, d)
        if attn_mask is not None:
            attn_mask = attn_mask.repeat(1, 1, n_rep, 1)
        out = BlockwiseAttention.apply(q, k, v, attn_mask, False, scale, block)
        return out.reshape(B, H, T, d)
    if n_rep > 1:
        k, v = k.repeat_interleave(n_rep, dim=1), v.repeat_interleave(n_rep, dim=1)
    return BlockwiseAttention.apply(q, k, v, attn_mask, is_causal, scale, block)


def _attend(q, k, v, causal_mask, key_keep, scale, attn_dropout, blockwise):
    """Causal scaled attention. q,k,v: (B,H,T,dh). key_keep: (B,T) bool or None (True = real token)."""
    T = q.size(-2)
    allowed = causal_mask[:T, :T]
    if key_keep is not None:                                   # CAPSTONE: left-padding support
        eye = torch.eye(T, dtype=torch.bool, device=q.device)  # a pad query may always see itself (avoids NaN rows)
        allowed = allowed & (key_keep[:, None, None, :] | eye)
    if blockwise:                                              # CAPSTONE: memory-efficient path, same result
        assert not (attn_dropout.training and attn_dropout.p > 0), "blockwise path has no attention dropout"
        if key_keep is None:
            return scratch_attention(q, k, v, is_causal=True, scale=scale)
        return scratch_attention(q, k, v, attn_mask=allowed, scale=scale)
    scores = torch.matmul(q, k.transpose(-2, -1)) * scale
    scores = scores.masked_fill(~allowed, float("-inf"))
    attn = attn_dropout(torch.softmax(scores, dim=-1))
    return torch.matmul(attn, v)


# ---- RoPE (A-3 Q3 spec: split-half pairing, theta_i = base^(-2i/d_head)) ----
def build_rope_cache(max_seq_len, d_head, base=10000.0):
    assert d_head % 2 == 0, "RoPE requires an even head dimension"
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2, dtype=torch.float32) / d_head))   # (d_head/2,)
    freqs = torch.outer(torch.arange(max_seq_len, dtype=torch.float32), inv_freq)           # (T, d_head/2)
    angles = torch.cat([freqs, freqs], dim=-1)                                               # (T, d_head)
    return angles.cos(), angles.sin()


def rotate_half(x):
    x1, x2 = x.chunk(2, dim=-1)
    return torch.cat([-x2, x1], dim=-1)


def apply_rope(x, cos, sin):
    """x: (B, H, T, d_head); cos/sin broadcastable to it, e.g. (1,1,T,d_head) or (B,1,T,d_head)."""
    return x * cos + rotate_half(x) * sin


class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0, attention_scale=None, blockwise_attn=False,
                 use_rope=False, rope_base=10000.0):
        super().__init__()
        assert d_model % n_heads == 0
        assert n_heads % n_kv_heads == 0, "n_heads must be divisible by n_kv_heads"
        self.n_heads, self.n_kv_heads = n_heads, n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.blockwise_attn = blockwise_attn
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)
        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("causal_mask", torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)), persistent=False)
        self.use_rope = use_rope
        if use_rope:                                           # cached cos/sin, not saved in the checkpoint
            cos, sin = build_rope_cache(max_seq_len, self.d_head, rope_base)
            self.register_buffer("rope_cos", cos, persistent=False)
            self.register_buffer("rope_sin", sin, persistent=False)

    def forward(self, x, key_keep=None, pos_ids=None):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        if self.use_rope:                                      # rotate q and k (before sharing kv heads)
            if pos_ids is None:
                cos, sin = self.rope_cos[:T][None, None], self.rope_sin[:T][None, None]
            else:                                              # per-row positions (left padding)
                cos, sin = self.rope_cos[pos_ids].unsqueeze(1), self.rope_sin[pos_ids].unsqueeze(1)
            q = apply_rope(q, cos.to(q.dtype), sin.to(q.dtype))
            k = apply_rope(k, cos.to(k.dtype), sin.to(k.dtype))
        if self.n_rep > 1:
            k = torch.repeat_interleave(k, self.n_rep, dim=1)
            v = torch.repeat_interleave(v, self.n_rep, dim=1)
        out = _attend(q, k, v, self.causal_mask, key_keep, self.attention_scale, self.attn_dropout, self.blockwise_attn)
        out = out.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class MultiHeadLatentAttention(nn.Module):
    def __init__(self, d_model, n_heads, kv_latent_dim, max_seq_len, dropout=0.0, attention_scale=None, blockwise_attn=False):
        super().__init__()
        if d_model % n_heads != 0:
            raise ValueError("d_model must be divisible by n_heads")
        if kv_latent_dim is None or not (0 < int(kv_latent_dim) < d_model):
            raise ValueError(f"kv_latent_dim must satisfy 0 < r < d_model, got {kv_latent_dim}")
        self.n_heads, self.d_head, self.kv_latent_dim = n_heads, d_model // n_heads, int(kv_latent_dim)
        self.blockwise_attn = blockwise_attn
        self.attention_scale = (1.0 / math.sqrt(self.d_head)) if attention_scale is None else float(attention_scale)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.kv_down_proj = nn.Linear(d_model, self.kv_latent_dim, bias=False)
        self.k_up_proj = nn.Linear(self.kv_latent_dim, d_model, bias=False)
        self.v_up_proj = nn.Linear(self.kv_latent_dim, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        self.register_buffer("causal_mask", torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool)), persistent=False)

    def forward(self, x, key_keep=None, pos_ids=None):    # pos_ids unused (MLA uses sinusoidal positions)
        B, T, D = x.shape
        H, dh = self.n_heads, self.d_head
        c = self.kv_down_proj(x)
        q = self.q_proj(x).view(B, T, H, dh).transpose(1, 2)
        k = self.k_up_proj(c).view(B, T, H, dh).transpose(1, 2)
        v = self.v_up_proj(c).view(B, T, H, dh).transpose(1, 2)
        out = _attend(q, k, v, self.causal_mask, key_keep, self.attention_scale, self.attn_dropout, self.blockwise_attn)
        out = out.transpose(1, 2).contiguous().view(B, T, D)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    pos = torch.arange(max_len, dtype=torch.float32).unsqueeze(1)
    div = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe = torch.zeros(max_len, d_model)
    pe[:, 0::2] = torch.sin(pos * div)
    pe[:, 1::2] = torch.cos(pos * div)[:, : d_model // 2]
    return pe


class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.net = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        return self.net(x)


class Block(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.norm_first = cfg.norm_first
        self.ln1 = nn.LayerNorm(cfg.d_model)
        if cfg.pos_emb_type not in ("sinusoidal", "rope"):
            raise ValueError("pos_emb_type must be 'sinusoidal' or 'rope'")
        if cfg.attention_type == "mla":
            if cfg.pos_emb_type != "sinusoidal":
                raise ValueError("MLA is restricted to sinusoidal positions (as in A-3 Q2)")
            self.attn = MultiHeadLatentAttention(cfg.d_model, cfg.n_heads, cfg.kv_latent_dim, cfg.max_seq_len,
                                                 cfg.dropout, cfg.attention_scale, cfg.blockwise_attn)
        elif cfg.attention_type == "standard":
            self.attn = CausalSelfAttention(cfg.d_model, cfg.n_heads, cfg.n_kv_heads, cfg.max_seq_len,
                                            cfg.dropout, cfg.attention_scale, cfg.blockwise_attn,
                                            use_rope=(cfg.pos_emb_type == "rope"), rope_base=cfg.rope_base)
        else:
            raise ValueError("attention_type must be 'standard' or 'mla'")
        self.ln2 = nn.LayerNorm(cfg.d_model)
        self.ffn = FeedForward(cfg.d_model, cfg.d_ff, cfg.dropout)

    def forward(self, x, key_keep=None, pos_ids=None):
        if self.norm_first:
            x = x + self.attn(self.ln1(x), key_keep, pos_ids)
            x = x + self.ffn(self.ln2(x))
        else:
            x = self.ln1(x + self.attn(x, key_keep, pos_ids))
            x = self.ln2(x + self.ffn(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.d_model)
        if cfg.pos_emb_type == "sinusoidal":
            self.register_buffer("pos_emb", sinusoidal_positional_encoding(cfg.max_seq_len, cfg.d_model), persistent=False)
        self.emb_drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layers)])
        self.ln_f = nn.LayerNorm(cfg.d_model)
        # CAPSTONE: weight tying is done in forward (logits = x @ tok_emb.weight^T) instead of
        # a separate `head` module sharing the tensor. Same math, but no shared tensors,
        # so Trainer checkpoints save cleanly as safetensors.
        self.apply(self._init_weights)

    @staticmethod
    def _init_weights(m):
        if isinstance(m, nn.Linear):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)
            if m.bias is not None:
                nn.init.zeros_(m.bias)
        elif isinstance(m, nn.Embedding):
            nn.init.normal_(m.weight, mean=0.0, std=0.02)

    def forward(self, idx, attention_mask=None):
        B, T = idx.shape
        assert T <= self.cfg.max_seq_len, f"sequence length {T} > max_seq_len {self.cfg.max_seq_len}"
        key_keep, pos_ids = None, None
        if attention_mask is not None and not bool(attention_mask.all()):
            key_keep = attention_mask.bool()                    # CAPSTONE: only when there is padding
            pos_ids = (key_keep.long().cumsum(-1) - 1).clamp(min=0)   # positions start at the first real token
        x = self.tok_emb(idx)
        if self.cfg.pos_emb_type == "sinusoidal":
            x = x + (self.pos_emb[:T].unsqueeze(0) if pos_ids is None else self.pos_emb[pos_ids])
        x = self.emb_drop(x)
        for block in self.blocks:
            x = block(x, key_keep, pos_ids)                     # RoPE (if used) is applied inside attention
        x = self.ln_f(x)
        return F.linear(x, self.tok_emb.weight)                 # tied output head


class A3ForCausalLM(nn.Module):
    """CAPSTONE: thin wrapper so Hugging Face Trainer can train the A-3 model.
    Takes input_ids / attention_mask / labels (from DataCollatorForLanguageModeling)
    and returns {'loss', 'logits'}. Labels are shifted here, like GPT2LMHeadModel."""

    def __init__(self, cfg: LMConfig):
        super().__init__()
        self.lm_config = cfg
        self.lm = DecoderLM(cfg)

    # No **kwargs on purpose: otherwise newer Trainer versions pass num_items_in_batch and
    # stop dividing the loss by gradient_accumulation_steps.
    def forward(self, input_ids, attention_mask=None, labels=None):
        logits = self.lm(input_ids, attention_mask=attention_mask)
        loss = None
        if labels is not None:
            loss = F.cross_entropy(logits[:, :-1].float().reshape(-1, logits.size(-1)),
                                   labels[:, 1:].reshape(-1), ignore_index=-100)
        return {"loss": loss, "logits": logits}


def save_a3_model(model, out_dir):
    model = model.module if hasattr(model, "module") else model
    os.makedirs(out_dir, exist_ok=True)
    torch.save(model.lm.state_dict(), os.path.join(out_dir, "a3_model.pt"))
    with open(os.path.join(out_dir, "a3_config.json"), "w") as f:
        json.dump(asdict(model.lm_config), f, indent=2)


def load_a3_model(out_dir, map_location="cpu", **overrides):
    with open(os.path.join(out_dir, "a3_config.json")) as f:
        saved = json.load(f)
        saved["blockwise_attn"] = saved.pop("blockwise_attn", saved.get("blockwise_attn", False))  # configs saved before the rename
        cfg = LMConfig(**{**saved, **overrides})
    model = A3ForCausalLM(cfg)
    model.lm.load_state_dict(torch.load(os.path.join(out_dir, "a3_model.pt"), map_location=map_location))
    return model


def _sample_top_p(logits, temperature, top_p):
    probs = torch.softmax(logits / temperature, dim=-1)
    sorted_probs, sorted_idx = torch.sort(probs, dim=-1, descending=True)
    cum = torch.cumsum(sorted_probs, dim=-1)
    sorted_probs[(cum - sorted_probs) > top_p] = 0.0          # keep the smallest set with mass >= top_p
    sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
    choice = torch.multinomial(sorted_probs, 1)
    return sorted_idx.gather(-1, choice).squeeze(-1)

## 9. Build the Model and Data Collator

The chosen configuration: 10 layers, d_model 768, 12 query heads sharing 4 key/value heads (GQA),
d_ff 3072, RoPE positions, pre-norm, 1024-token context, which comes to **~101M parameters**.

`DataCollatorForLanguageModeling(mlm=False)` turns each batch of blocks into `input_ids`,
`attention_mask` and `labels` (a copy of the inputs; the model shifts them internally for
next-token prediction).

In [ ]:
config = LMConfig(
    vocab_size=len(tokenizer),
    d_model=768,
    n_layers=10,
    n_heads=12,
    n_kv_heads=4,
    d_ff=3072,
    max_seq_len=MODEL_CONTEXT_LENGTH,
    dropout=0.0,
    pos_emb_type="rope",
    norm_first=True,
    attention_type="standard",
    blockwise_attn=True,
)
model = A3ForCausalLM(config)
data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)
print(f"Model size: {sum(p.numel() for p in model.parameters()) / 1e6:.2f}M parameters")   # expect ~101.4M

## 10. Training

**Training arguments:** effective batch = 4 sequences x 2 accumulation steps x number of GPUs; AdamW at
lr 3e-4 with 500 warm-up steps and cosine decay; fp16 mixed precision on GPU. The model is evaluated
every 1000 steps and checkpointed every 1000 steps (only the last 2 checkpoints are kept).

`prediction_loss_only=True` makes evaluation compute only the loss. Otherwise `Trainer` would keep
every logit (validation tokens x 50k vocab) in memory and crash.

`TimeLimit` is a safety callback: it stops training cleanly after `TIME_LIMIT_HOURS`, so there is
always time left to save the model and generate predictions before Kaggle's 12-hour limit.

In [ ]:
training_args = TrainingArguments(
    output_dir=MODEL_OUTPUT_DIR,
    num_train_epochs=EPOCHS,
    max_steps=MAX_STEPS,
    per_device_train_batch_size=4,
    per_device_eval_batch_size=4,
    gradient_accumulation_steps=2,
    learning_rate=3e-4,
    weight_decay=0.1,
    warmup_steps=500,
    lr_scheduler_type="cosine",
    fp16=torch.cuda.is_available(),
    eval_strategy="steps",
    eval_steps=EVAL_STEPS,
    logging_steps=100,
    save_steps=SAVE_STEPS,
    save_total_limit=2,
    prediction_loss_only=True,
    seed=SEED,
    report_to="none",
)


class TimeLimit(TrainerCallback):
    """Safety net: stop training after a fixed wall-clock budget."""
    def __init__(self, seconds):
        self.seconds = seconds
    def on_train_begin(self, args, state, control, **kwargs):
        self.t0 = time.time()
    def on_step_end(self, args, state, control, **kwargs):
        if time.time() - self.t0 > self.seconds:
            control.should_training_stop = True


trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    data_collator=data_collator,
    processing_class=tokenizer,
    callbacks=[TimeLimit(TIME_LIMIT_HOURS * 3600)],
)

start_time = time.time()
trainer.train()
training_time = time.time() - start_time

# counted from the steps actually run (correct even if TimeLimit stopped training early)
tokens_seen = trainer.state.global_step * training_args.train_batch_size * training_args.gradient_accumulation_steps * TRAIN_BLOCK_SIZE
print(f"Steps: {trainer.state.global_step:,} | Training time: {training_time / 3600:.2f} hours")
print(f"Tokens seen: {tokens_seen:,} ({tokens_seen / training_time:,.0f} tokens/sec)")

## 11. Validation Perplexity and Loss Curves

Perplexity = exp(validation cross-entropy loss): roughly, how many tokens the model is "choosing
between" on average. Lower is better. The plot shows training loss (every 100 steps) and validation
loss (every 1000 steps); it is saved as `training_validation_loss.png`.

In [ ]:
eval_loss = trainer.evaluate()["eval_loss"]
print(f"Validation loss: {eval_loss:.6f}")
print(f"Validation perplexity: {math.exp(eval_loss):.4f}")

logs = trainer.state.log_history
train_log = [(l["step"], l["loss"]) for l in logs if "loss" in l]
eval_log = [(l["step"], l["eval_loss"]) for l in logs if "eval_loss" in l]

plt.figure(figsize=(10, 5))
if train_log:
    plt.plot(*zip(*train_log), label="Training Loss")
if eval_log:
    plt.plot(*zip(*eval_log), label="Validation Loss")
plt.xlabel("Training Steps")
plt.ylabel("Loss")
plt.title("Phase 1 Training and Validation Loss")
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.savefig(LOSS_PLOT_PATH, dpi=200)
plt.show()
print("Loss plot saved to:", LOSS_PLOT_PATH)

## 12. Save the Final Model

`save_a3_model` writes the weights (`a3_model.pt`) and architecture (`a3_config.json`), and
`save_pretrained` writes the tokenizer to the same folder. Together they fully describe the model,
so it can be reloaded later with `load_a3_model` + `LegalBPETokenizer.from_pretrained`.

In [ ]:
save_a3_model(trainer.model, MODEL_OUTPUT_DIR)
tokenizer.save_pretrained(MODEL_OUTPUT_DIR)
print("Final model saved to:", MODEL_OUTPUT_DIR)

## 13. Load Test Prompts and the Saved Model

Downloads the test file (a dict of `id -> {"text": prompt}`) and reloads the model from disk. This
confirms the saved files are complete and gives a clean, single-GPU copy for generation.

In [ ]:
test_file_path = hf_hub_download(repo_id=HF_DATASET_REPO, filename=TEST_FILE, repo_type="dataset", token=HF_TOKEN)
with open(test_file_path, "r", encoding="utf-8") as f:
    test_data = json.load(f)
if NUM_TEST_EXAMPLES:
    test_data = dict(list(test_data.items())[:NUM_TEST_EXAMPLES])
print("Number of test examples:", len(test_data))

generation_tokenizer = LegalBPETokenizer.from_pretrained(MODEL_OUTPUT_DIR)
generation_model = load_a3_model(MODEL_OUTPUT_DIR, blockwise_attn=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
generation_model = generation_model.to(device).eval()
print("Device:", device)
print(f"Parameters: {sum(p.numel() for p in generation_model.parameters()) / 1e6:.2f}M")

## 14. Batched Text Generation (128 words, not tokens)

`generate_batch` continues a batch of prompts:
1. Each prompt is encoded with `encode_fast` (the same tokenization used in training) and
   **left-truncated**, keeping its end, so prompt + generation fits in the 1024-token context.
2. Prompts are **left-padded** to the same length, and the attention mask hides the padding.
3. Tokens are sampled one at a time with temperature + top-p (`_sample_top_p`) under fp16 autocast.
4. Generation stops early once every row has hit `</s>` or already has more than 128 words. Each
   output is then cut to exactly the first 128 words.

In [ ]:
def generate_batch(prompts, model, tokenizer, device, target_words=128, check_every=16):
    # Tokenize with encode_fast: the SAME function used to build the training tokens.
    # Then left-truncate (keep the end of each prompt) and left-pad by hand.
    max_prompt = MODEL_CONTEXT_LENGTH - MAX_GENERATION_TOKENS
    eos_id, pad_id = tokenizer.eos_token_id, tokenizer.pad_token_id
    seqs = []
    for p in prompts:
        text = p if isinstance(p, str) else ("" if p is None else str(p))
        ids = tokenizer.encode_fast(text)[-max_prompt:]
        seqs.append(ids if ids else [eos_id])      # empty prompt -> start like a new document
    L = max(len(s) for s in seqs)
    input_ids = torch.tensor([[pad_id] * (L - len(s)) + s for s in seqs], dtype=torch.long, device=device)
    attention_mask = torch.tensor([[0] * (L - len(s)) + [1] * len(s) for s in seqs], dtype=torch.long, device=device)
    prompt_len = input_ids.size(1)
    finished = torch.zeros(input_ids.size(0), dtype=torch.bool, device=device)
    ctx = model.lm.cfg.max_seq_len
    use_amp = device.type == "cuda"

    with torch.inference_mode():
        for step in range(MAX_GENERATION_TOKENS):
            with torch.autocast(device_type=device.type, dtype=torch.float16, enabled=use_amp):
                logits = model.lm(input_ids[:, -ctx:], attention_mask=attention_mask[:, -ctx:])[:, -1, :]
            next_tok = _sample_top_p(logits.float(), TEMPERATURE, TOP_P)
            next_tok = torch.where(finished, torch.full_like(next_tok, eos_id), next_tok)
            input_ids = torch.cat([input_ids, next_tok[:, None]], dim=1)
            attention_mask = torch.cat([attention_mask, torch.ones_like(next_tok[:, None])], dim=1)
            finished |= next_tok == eos_id

            # Stop early once every row is done or already has more than target_words words.
            if finished.all():
                break
            if (step + 1) % check_every == 0:
                texts = tokenizer.batch_decode(input_ids[:, prompt_len:], skip_special_tokens=True)
                if all(f or len(t.split()) > target_words for f, t in zip(finished.tolist(), texts)):
                    break

    predictions = []
    for i in range(len(prompts)):
        generated_text = tokenizer.decode(input_ids[i, prompt_len:], skip_special_tokens=True)
        predictions.append(" ".join(generated_text.split()[:target_words]))
    return predictions

## 15. Generate Predictions for the Whole Test Set

The test prompts are processed in batches of `GENERATION_BATCH_SIZE`, and each prediction is stored
as `{id: {"text": continuation}}`, the format CodaBench expects.

In [ ]:
predictions = {}
items = list(test_data.items())
total_examples = len(items)

for start in range(0, total_examples, GENERATION_BATCH_SIZE):
    batch = items[start:start + GENERATION_BATCH_SIZE]
    batch_predictions = generate_batch(
        prompts=[record["text"] for _, record in batch],
        model=generation_model,
        tokenizer=generation_tokenizer,
        device=device,
        target_words=TARGET_GENERATION_WORDS,
    )
    for (example_id, _), prediction in zip(batch, batch_predictions):
        predictions[str(example_id)] = {"text": prediction}

    completed = start + len(batch)
    print(f"Completed {completed}/{total_examples} ({100 * completed / total_examples:.1f}%)")

## 16. Save `prediction.json` and the CodaBench Submission

Writes the predictions to `prediction.json` and zips them into `submission.zip`. Download the zip
from the notebook's **Output** tab and upload it to CodaBench.

In [ ]:
with open(PREDICTION_PATH, "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

with zipfile.ZipFile(SUBMISSION_ZIP, "w", compression=zipfile.ZIP_DEFLATED) as zipf:
    zipf.write(PREDICTION_PATH, arcname="prediction.json")

print("prediction.json saved:", PREDICTION_PATH)
print("Codabench submission created:", SUBMISSION_ZIP)